In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("data/r2_cleaned.csv")
df.head()

,AGENT,VIDEO,BLOCK,QUESTION_NUM,REPETITION,ANSWER
0,human_lima_1,Robusto2_153,1,1,1,The ego vehicle is accelerating slowly because...
1,human_lima_2,Robusto2_153,1,1,1,The ego vehicle is turning to the right
2,human_lima_3,Robusto2_153,1,1,1,the ego vehicle brakes and steers slightly to ...
3,human_lima_4,Robusto2_153,1,1,1,Braking to yield
4,human_lima_5,Robusto2_153,1,1,1,The ego vehicle is moving forward while mainta...


In [4]:
def video_name(video_id):
    return f"Robusto2_{video_id}"

def query(df, video_id, question_number):
    video = video_name(video_id)
    result = df[(df['VIDEO'] == video) & (df['QUESTION_NUM'] == question_number)]
    if not result.empty:
        return result

d = query(df, 62, 1)
d

,AGENT,VIDEO,BLOCK,QUESTION_NUM,REPETITION,ANSWER
3000,human_lima_1,Robusto2_62,1,1,1,The ego vehicle was accelerating but then stopped
3001,human_lima_2,Robusto2_62,1,1,1,The ego vehicle is making a slow move forward.
3002,human_lima_3,Robusto2_62,1,1,1,the ego vehicle is braking
3003,human_lima_4,Robusto2_62,1,1,1,Stopped in traffic
3004,human_lima_5,Robusto2_62,1,1,1,The ego vehicle is stopped in its lane due to ...
...,...,...,...,...,...,...
80815,VideoLLaMA3-7B,Robusto2_62,1,1,16,The ego vehicle is stationary.
80816,VideoLLaMA3-7B,Robusto2_62,1,1,17,The ego vehicle is stationary.
80817,VideoLLaMA3-7B,Robusto2_62,1,1,18,The ego vehicle is stationary.
80818,VideoLLaMA3-7B,Robusto2_62,1,1,19,The ego vehicle is stationary.


In [5]:
#multi query



In [9]:
import textwrap
import random
import re
import subprocess
import io
import json
import yaml
from pathlib import Path
from PIL import Image


VIDEO_ROOT = Path("final_videos")          # contains bloq1/.../mp4
FRAMES_OUT = Path("outputs/paper_frames")  # where extracted PNGs are saved


def load_questions_yaml(path="final_questions_v3.yaml"):
    """Load per-video question text. Returns {video_name: {q_num: text}}."""
    with open(path, "r", encoding="utf-8") as f:
        raw = yaml.safe_load(f)
    out = {}
    for video, qs in raw.items():
        out[video] = {}
        for qkey, meta in qs.items():
            m = re.match(r"Q(\d+)$", qkey)
            if not m:
                continue
            out[video][int(m.group(1))] = meta.get("question", "").strip()
    return out


def _index_video_files(root=VIDEO_ROOT):
    idx = {}
    if not root.exists():
        return idx
    for mp4 in root.glob("bloq*/*.mp4"):
        idx[mp4.stem] = mp4
    return idx


QUESTIONS_BY_VIDEO = load_questions_yaml(Path("final_questions_v3.yaml"))
VIDEO_FILES = _index_video_files()
print(f"Loaded questions for {len(QUESTIONS_BY_VIDEO)} videos.")
print(f"Indexed {len(VIDEO_FILES)} mp4 files in {VIDEO_ROOT}/.")


VLM_AGENTS = {
    "Cosmos-Reason2-8B", "Gemini3-Flash-preview", "Gemini3-Pro-preview",
    "InternVL3-8B", "LLaVA-Video-7B-Qwen2", "MiniCPM-o-2_6",
    "Perception-LM-8B", "Phi-4-multimodal-instruct",
    "Qwen3-VL-8B-Instruct", "VideoLLaMA3-7B",
}


def _agent_group(agent):
    a = agent.lower()
    if a.startswith("human_lima"):
        return "human_lima"
    if a.startswith("human_nyc"):
        return "human_nyc"
    if agent in VLM_AGENTS:
        return "vlm"
    return "other"


def list_agents():
    print("human_lima:", [f"human_lima_{i}" for i in range(1, 11)])
    print("human_nyc :", [f"human_nyc_{i}" for i in range(11, 16)])
    print("vlm       :", sorted(VLM_AGENTS))


def list_videos(df):
    ids = []
    for v in df["VIDEO"].unique():
        m = re.search(r"_(\d+)$", str(v))
        if m:
            ids.append(int(m.group(1)))
    ids = sorted(set(ids))
    print(f"videos ({len(ids)}):", ids)
    return ids


def question_text(video, q):
    return QUESTIONS_BY_VIDEO.get(video, {}).get(int(q), f"Q{int(q)}")


def find_video_path(video_id):
    """Return the Path to Robusto2_<video_id>.mp4 inside final_videos/bloq*/, or None."""
    return VIDEO_FILES.get(video_name(video_id))


def _video_duration(path):
    cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration",
           "-of", "json", str(path)]
    return float(json.loads(subprocess.check_output(cmd))["format"]["duration"])


def save_video_frames(video_id, n=6, out_dir=FRAMES_OUT):
    """Extract `n` evenly-spaced frames and write them as PNG files. No notebook display.

    Files are written to out_dir/<video>/frame_{idx:02d}_t{seconds}.png.
    Returns the list of saved Paths (also printed for easy copy-paste).
    """
    path = find_video_path(video_id)
    if path is None:
        print(f"[warn] {video_name(video_id)}.mp4 not found under {VIDEO_ROOT}/")
        return []

    dur = _video_duration(path)
    if n == 1:
        ts = [dur / 2]
    else:
        step = (dur * 0.9) / (n - 1)
        ts = [dur * 0.05 + i * step for i in range(n)]

    target = out_dir / video_name(video_id)
    target.mkdir(parents=True, exist_ok=True)
    saved = []
    for i, t in enumerate(ts, start=1):
        out_file = target / f"frame_{i:02d}_t{t:05.2f}s.png"
        cmd = ["ffmpeg", "-y", "-loglevel", "error",
               "-ss", f"{t:.3f}", "-i", str(path),
               "-frames:v", "1", str(out_file)]
        subprocess.check_call(cmd)
        saved.append(out_file)

    print(f"saved {len(saved)} frames to {target}/")
    for p in saved:
        print(f"  {p}")
    return saved


def _select_group(sub, group, n, rng, pinned=None):
    g = sub[sub["AGENT"].apply(_agent_group) == group]
    if g.empty:
        return g
    if group == "vlm":
        g = g[g["REPETITION"] == 1]

    if pinned:
        missing = [a for a in pinned if a not in set(g["AGENT"])]
        if missing:
            print(f"  [warn] {group}: agents not found for this row -> {missing}")
        picks = [a for a in pinned if a in set(g["AGENT"])]
    else:
        picks = list(g["AGENT"].unique())
        rng.shuffle(picks)
        picks = picks[:n]

    rows = g[g["AGENT"].isin(picks)].drop_duplicates(subset=["AGENT"])
    order = {a: i for i, a in enumerate(picks)}
    return rows.assign(_o=rows["AGENT"].map(order)).sort_values("_o").drop(columns="_o")


def figure_samples(
    df,
    video_id,
    questions_per_block=(1, 2, 6, 7, 11, 12, 16, 17),
    n_per_group=2,
    seed=0,
    agents=None,
):
    """Build a figure-ready dict for a single video. Includes resolved video path."""
    rng = random.Random(seed)
    agents = agents or {}
    video = video_name(video_id)
    d = df[df["VIDEO"] == video]
    if d.empty:
        raise ValueError(f"No rows for {video}. Available: {list_videos(df)}")

    vpath = find_video_path(video_id)
    if vpath is None:
        print(f"[warn] no mp4 found for {video} under {VIDEO_ROOT}/")

    out = {}
    for q in questions_per_block:
        sub = d[d["QUESTION_NUM"] == q]
        if sub.empty:
            continue
        block = int(sub["BLOCK"].iloc[0])
        entry = {
            "video": video,
            "video_path": str(vpath) if vpath else None,
            "question_num": int(q),
            "question_text": question_text(video, q),
            "groups": {},
        }
        for grp in ("human_lima", "human_nyc", "vlm"):
            picks = _select_group(sub, grp, n_per_group, rng, pinned=agents.get(grp))
            entry["groups"][grp] = [(r["AGENT"], r["ANSWER"]) for _, r in picks.iterrows()]
        out.setdefault(block, []).append(entry)
    return out


def print_figure_samples(samples, width=100):
    wrap_ans = lambda s: textwrap.fill(str(s), width=width, subsequent_indent=" " * 6)
    wrap_q = lambda s: textwrap.fill(str(s), width=width, subsequent_indent=" " * 5)
    any_entry = next((e for v in samples.values() for e in v), None)
    if any_entry:
        print(f"VIDEO: {any_entry['video']}")
        print(f"FILE : {any_entry['video_path']}")
    for block in sorted(samples):
        print(f"\n{'=' * width}\nBLOCK {block}\n{'=' * width}")
        for entry in samples[block]:
            print(f"\n[Q{entry['question_num']}] {wrap_q(entry['question_text'])}")
            for grp, rows in entry["groups"].items():
                print(f"  -- {grp} --")
                if not rows:
                    print("     (no data)")
                    continue
                for agent, ans in rows:
                    print(f"     [{agent}]")
                    print(f"      {wrap_ans(ans)}")


# --- Configuration: change these once, used everywhere below ---------------------
list_videos(df)
list_agents()

VIDEO_ID = 62
QUESTIONS = (1, 2, 6, 7, 11, 12, 16, 17)  # 2 per block × 4 blocks
N_FRAMES = 6
AGENTS = {
    "human_lima": ["human_lima_1","human_lima_6"],
    "human_nyc":  ["human_nyc_12","human_nyc_15"],
    "vlm":        ["Cosmos-Reason2-8B","Gemini3-Flash-preview"],
}

samples = figure_samples(df, video_id=VIDEO_ID, questions_per_block=QUESTIONS, agents=AGENTS)
print_figure_samples(samples)
save_video_frames(VIDEO_ID, n=N_FRAMES)

Loaded questions for 200 videos.
Indexed 20 mp4 files in final_videos/.
videos (20): [18, 22, 37, 40, 46, 56, 58, 62, 69, 82, 113, 115, 131, 153, 171, 180, 181, 182, 192, 195]
human_lima: ['human_lima_1', 'human_lima_2', 'human_lima_3', 'human_lima_4', 'human_lima_5', 'human_lima_6', 'human_lima_7', 'human_lima_8', 'human_lima_9', 'human_lima_10']
human_nyc : ['human_nyc_11', 'human_nyc_12', 'human_nyc_13', 'human_nyc_14', 'human_nyc_15']
vlm       : ['Cosmos-Reason2-8B', 'Gemini3-Flash-preview', 'Gemini3-Pro-preview', 'InternVL3-8B', 'LLaVA-Video-7B-Qwen2', 'MiniCPM-o-2_6', 'Perception-LM-8B', 'Phi-4-multimodal-instruct', 'Qwen3-VL-8B-Instruct', 'VideoLLaMA3-7B']
VIDEO: Robusto2_62
FILE : final_videos\bloq3\Robusto2_62.mp4

BLOCK 1

[Q1] What action is the ego vehicle taking?
  -- human_lima --
     [human_lima_1]
      The ego vehicle was accelerating but then stopped
     [human_lima_6]
      It is stationary, waiting in a dense traffic queue
  -- human_nyc --
     [human_nyc_12]
  

[WindowsPath('outputs/paper_frames/Robusto2_62/frame_01_t00.25s.png'),
 WindowsPath('outputs/paper_frames/Robusto2_62/frame_02_t01.15s.png'),
 WindowsPath('outputs/paper_frames/Robusto2_62/frame_03_t02.05s.png'),
 WindowsPath('outputs/paper_frames/Robusto2_62/frame_04_t02.95s.png'),
 WindowsPath('outputs/paper_frames/Robusto2_62/frame_05_t03.85s.png'),
 WindowsPath('outputs/paper_frames/Robusto2_62/frame_06_t04.75s.png')]

In [8]:
VIDEO_ID = 192
samples = figure_samples(df, video_id=VIDEO_ID, questions_per_block=QUESTIONS, agents=AGENTS)
print_figure_samples(samples)
save_video_frames(VIDEO_ID, n=N_FRAMES)

VIDEO: Robusto2_192
FILE : final_videos\bloq2\Robusto2_192.mp4

BLOCK 1

[Q1] What is the ego vehicle’s action?
  -- human_lima --
     [human_lima_3]
      the ego vehicle is moving forward, but it braked slightly and steered a little to the right for
      safety
  -- human_nyc --
     [human_nyc_11]
      It is braking while crossing the intersection
  -- vlm --
     [Cosmos-Reason2-8B]
      The ego vehicle is cautiously merging left to avoid a traffic jam ahead.

[Q2] Why is the ego vehicle braking?
  -- human_lima --
     [human_lima_3]
      because a car from the left is entering its lane
  -- human_nyc --
     [human_nyc_11]
      Because a car ahead is entering to its lane.
  -- vlm --
     [Cosmos-Reason2-8B]
      The ego vehicle is braking because there are vehicles ahead that are also braking, indicating a need
      to slow down to maintain a safe distance.

BLOCK 2

[Q6] Please rate the level of clutter from 1 to 10. Consider 10 as the highest level of clutter and 1 as


[WindowsPath('outputs/paper_frames/Robusto2_192/frame_01_t00.25s.png'),
 WindowsPath('outputs/paper_frames/Robusto2_192/frame_02_t01.15s.png'),
 WindowsPath('outputs/paper_frames/Robusto2_192/frame_03_t02.05s.png'),
 WindowsPath('outputs/paper_frames/Robusto2_192/frame_04_t02.95s.png'),
 WindowsPath('outputs/paper_frames/Robusto2_192/frame_05_t03.85s.png'),
 WindowsPath('outputs/paper_frames/Robusto2_192/frame_06_t04.75s.png')]